# **Memory**

## **TODO:**
check out the following for this notebook:
- sql tool project: https://academy.langchain.com/courses/take/langchain-essentials-python/lessons/69388329-lesson-6-memory
- (Middleware based) Accessing memory from the Tools, Prompts, Before Model and After Model: https://docs.langchain.com/oss/python/langchain/short-term-memory#access-memory
- (Middleware based) Common Pattern: Trim, Delete and Summarize Messages
- (LangGraph based) Manage Messages inside Memory: https://docs.langchain.com/oss/python/langgraph/add-memory#manage-short-term-memory
- (LangGraph based) Long term and Short term Memory: https://docs.langchain.com/oss/python/concepts/memory
- (Middleware based) Accessing Dynamic Short-term and Long-term memory: https://docs.langchain.com/oss/python/concepts/context
- Database management: https://docs.langchain.com/oss/python/langgraph/add-memory#database-management

Idea:
- Whenever a message arrive: run a batch job to save cost
- This batch job can invoke LLM on the user query as well as summarize and perform other reasoning steps as well.

## **Static Runtime Context**

1. Specify the **context_schema**: This defines the structure of context stored in the agent runtime
2. Provide the context_schema to the agent during **create_agent()**
3. In the tool call, use the **runtime** parameter (typed as **ToolRuntime**).
4. Finally, when you invoke the agent, you can provide your dependencies like db connection to the agent in the **context** argument.

Reference: https://docs.langchain.com/oss/python/langchain/short-term-memory

### **Define Agent State with middleware**

Use middleware to define custom state when your custom state needs to be accessed by specific middleware hooks and tools attached to said middleware.

https://docs.langchain.com/oss/python/langchain/agents#memory

## **EXPERIMENTAL: Agent State with Preferences Dictionary**

In [23]:
from langchain.agents import create_agent, AgentState
from langchain.tools import tool, ToolRuntime
from langgraph.checkpoint.memory import InMemorySaver

from langgraph.types import Command
from langchain.messages import ToolMessage
from typing import Dict
from langgraph.config import get_stream_writer

class CustomAgentState(AgentState):  
    user_id: str
    preferences: Dict[str, str]

@tool
def get_user_info(
    user_id: str,
    runtime: ToolRuntime
) -> str:
    """Look up user info."""
    uid = runtime.state.get("user_id")     # this will fetch the user_id from the custom state
    return f"User preferences are: {runtime.state.get('preferences')}" if user_id == uid else "Unknown user"

@tool
def update_preferences(
    runtime: ToolRuntime,
    **preference: dict,  # {"favorite_color": "blue"}
) -> Command:
    """
    Update the preferences of the user in the state once they've revealed it. 
    Preferences can be favourite color, language, etc...
    Pass dict like {'favorite_color': 'blue'}.
    
    Examples:
    - {'favorite_color': 'blue'}
    - {'language': 'French', 'theme': 'dark'}
    """
    writer = get_stream_writer()
    
    writer("Updating user preferences...")
    writer(f"Looking up data for preference in user input: {preference}")
    current_prefs = runtime.state.get("preferences", {})
    writer(f"Previous prefs: {current_prefs}")
    current_prefs.update(preference)
    writer(f"Updated prefs: {current_prefs}")
    
    return Command(
        update={
            "preferences": current_prefs,
            "messages": [ToolMessage(f"Updated preferences successfully.", tool_call_id=runtime.tool_call_id)]
        }
    )

agent = create_agent(
    model=openai_chat_model,
    tools=[get_user_info, update_preferences],
    checkpointer=InMemorySaver(),
    state_schema=CustomAgentState,
    
)

In [24]:
for mode_chunk in agent.stream(  
    {
        "messages": [{"role": "user", "content": "hi"}],
        "user_id": "user_123"
    },
    {
        "configurable" : {"thread_id" : "1"}
    },
    stream_mode=["values", "custom"],
):
    mode, chunk = mode_chunk  # ✅ Unpack tuple (mode, chunk)
    # if mode == "values":
    #     # Full state updates (agent steps)
    #     print("📊 State update:")
    #     for key, value in chunk.items():
    #         print(f"  {key}: {value}")
    #     print()
    
    if mode == "custom":
        # ✅ Custom tool streams
        print("🛠️  Tool stream:", chunk)
        print()
    
    # Pretty print last message if present
    if "messages" in chunk and chunk["messages"]:
        chunk["messages"][-1].pretty_print()

================================ Human Message =================================

hi
================================== Ai Message ==================================

Hello! How can I assist you today?


In [25]:
# response = agent.invoke(
#     {
#         "messages": [{"role": "user", "content": "Hi"}],
#         "user_id": "user_123"
#     },
#     {
#         "configurable" : {"thread_id" : "1"}
#     }                    
# )

# for msg in response["messages"]:
#     msg.pretty_print()

In [26]:
for mode_chunk in agent.stream(  
    {
        "messages": [{"role": "user", "content": "get me user information for user_123"}],
        "user_id": "user_123"
    },
    {
        "configurable" : {"thread_id" : "1"}
    },
    stream_mode=["values", "custom"],
):
    mode, chunk = mode_chunk  # ✅ Unpack tuple (mode, chunk)
    # if mode == "values":
    #     # Full state updates (agent steps)
    #     print("📊 State update:")
    #     for key, value in chunk.items():
    #         print(f"  {key}: {value}")
    #     print()
    
    if mode == "custom":
        # ✅ Custom tool streams
        print("🛠️  Tool stream:", chunk)
        print()
    
    # Pretty print last message if present
    if "messages" in chunk and chunk["messages"]:
        chunk["messages"][-1].pretty_print()

================================ Human Message =================================

get me user information for user_123
================================== Ai Message ==================================
Tool Calls:
  get_user_info (call_aZdGo3jZshMrpgIMny2VkzUN)
 Call ID: call_aZdGo3jZshMrpgIMny2VkzUN
  Args:
    user_id: user_123
================================= Tool Message =================================
Name: get_user_info

User preferences are: None
================================== Ai Message ==================================

It appears that there is no specific information or preferences available for user_123. If you need anything else or want to update preferences, just let me know!


In [21]:
# response = agent.invoke(
#     {
#         "messages": [{"role": "user", "content": "get me user information for user_123"}],
#         "user_id": "user_123"
#     },
#     {
#         "configurable" : {"thread_id" : "1"}
#     }                    
# )

# for msg in response["messages"]:
#     msg.pretty_print()

In [27]:
for mode_chunk in agent.stream(  
    {
        "messages": [{"role": "user", "content": "My favorite color is blue and I speak French"}],
        "user_id": "user_123"
    },
    {
        "configurable" : {"thread_id" : "1"}
    },
    stream_mode=["values", "custom"],
):
    mode, chunk = mode_chunk  # ✅ Unpack tuple (mode, chunk)
    # if mode == "values":
    #     # Full state updates (agent steps)
    #     print("📊 State update:")
    #     for key, value in chunk.items():
    #         print(f"  {key}: {value}")
    #     print()
    
    if mode == "custom":
        # ✅ Custom tool streams
        print("🛠️  Tool stream:", chunk)
        print()
    
    # Pretty print last message if present
    if "messages" in chunk and chunk["messages"]:
        chunk["messages"][-1].pretty_print()

================================ Human Message =================================

My favorite color is blue and I speak French
================================== Ai Message ==================================
Tool Calls:
  update_preferences (call_Am3G55rlSl34dsffMl3186uU)
 Call ID: call_Am3G55rlSl34dsffMl3186uU
  Args:
🛠️  Tool stream: Updating user preferences...

🛠️  Tool stream: Looking up data for preference in user input: {'preference': None}

🛠️  Tool stream: Previous prefs: {}

🛠️  Tool stream: Updated prefs: {'preference': None}

================================= Tool Message =================================
Name: update_preferences

Updated preferences successfully.
================================== Ai Message ==================================

Your preferences have been updated! Your favorite color is now blue, and you speak French. If there's anything else you'd like to add or change, just let me know!


In [5]:
response = agent.invoke(
    {
        "messages": [{"role": "user", "content": "My favorite color is blue and I speak French"}],
        "user_id": "user_123"
    },
    {
        "configurable" : {"thread_id" : "1"}
    }                    
)

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

Hi
================================== Ai Message ==================================

Hello! How can I assist you today?
================================ Human Message =================================

get me user information for user_123
================================== Ai Message ==================================
Tool Calls:
  get_user_info (call_CeSAVFPsSzVMaouQGMhQsqbb)
 Call ID: call_CeSAVFPsSzVMaouQGMhQsqbb
  Args:
    user_id: user_123
================================= Tool Message =================================
Name: get_user_info

User preferences are: None
================================== Ai Message ==================================

It looks like there are no preferences set for user_123. If you need any specific information or would like to set preferences, just let me know!
================================ Human Message =================================

My favorite color is blue and

In [6]:
response.keys()

dict_keys(['messages', 'user_id'])